In [1]:
# ============================================================
# STEP 7 - CELL 1
# SETUP
# ============================================================

import os
import json
import pickle
import time
import warnings

from pathlib import Path

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

INPUT_DIR = Path("/kaggle/input")

OUTPUT_DIR = Path("/kaggle/working/step7")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

pd.set_option(
    "display.max_columns",
    50
)

pd.set_option(
    "display.max_colwidth",
    100
)

print("=" * 70)
print("STEP 7 - HYBRID RECOMMENDER")
print("=" * 70)

print("Input :", INPUT_DIR)
print("Output:", OUTPUT_DIR)

print("\n✅ Setup completed")

STEP 7 - HYBRID RECOMMENDER
Input : /kaggle/input
Output: /kaggle/working/step7

✅ Setup completed


In [2]:
# ============================================================
# STEP 7 - CELL 2
# DISCOVER INPUT ARTIFACTS
# ============================================================

print("=" * 70)
print("DISCOVER INPUT ARTIFACTS")
print("=" * 70)

REQUIRED_FILES = [
    "products.csv",
    "reviews.csv",
    "interactions.csv",
    "cf_interactions.csv",

    "product_embeddings.npy",

    "bpr_model.pkl",
    "cf_mappings.pkl",
    "cf_train.csv",
    "cf_config.json"
]

OPTIONAL_FILES = [
    "product_embedding_metadata.csv",
    "product_metadata.csv",
    "semantic_products.csv",

    "clean_products.csv",
    "clean_reviews.csv",

    "bpr_validation_results.csv",
    "cf_test_results.csv",
    "bpr_full_test_results.csv"
]


def find_artifact(filename):

    matches = sorted(
        INPUT_DIR.rglob(filename)
    )

    if not matches:
        return None

    if len(matches) > 1:

        print(
            f"\n⚠ Multiple files named {filename}:"
        )

        for path in matches:
            print(" ", path)

        print(
            "Using:",
            matches[0]
        )

    return matches[0]


artifact_paths = {}

for filename in REQUIRED_FILES + OPTIONAL_FILES:

    path = find_artifact(filename)

    artifact_paths[filename] = path


print("\n[1] REQUIRED FILES")
print("-" * 70)

missing_required = []

for filename in REQUIRED_FILES:

    path = artifact_paths[filename]

    if path is None:

        print(f"❌ {filename}")

        missing_required.append(
            filename
        )

    else:

        print(
            f"✅ {filename:<35} {path}"
        )


print("\n[2] OPTIONAL FILES")
print("-" * 70)

for filename in OPTIONAL_FILES:

    path = artifact_paths[filename]

    if path is not None:

        print(
            f"✅ {filename:<35} {path}"
        )


print("\n" + "=" * 70)

if missing_required:

    print("MISSING REQUIRED FILES:")

    for filename in missing_required:

        print(" -", filename)

else:

    print("✅ All required artifacts found")

DISCOVER INPUT ARTIFACTS

[1] REQUIRED FILES
----------------------------------------------------------------------
✅ products.csv                        /kaggle/input/datasets/trnquangtriu/7dataset/products.csv
✅ reviews.csv                         /kaggle/input/datasets/trnquangtriu/7dataset/reviews.csv
✅ interactions.csv                    /kaggle/input/datasets/trnquangtriu/7dataset/interactions.csv
✅ cf_interactions.csv                 /kaggle/input/datasets/trnquangtriu/7dataset/cf_interactions.csv
✅ product_embeddings.npy              /kaggle/input/datasets/trnquangtriu/7dataset/product_embeddings.npy
✅ bpr_model.pkl                       /kaggle/input/datasets/trnquangtriu/7dataset/bpr_model.pkl
✅ cf_mappings.pkl                     /kaggle/input/datasets/trnquangtriu/7dataset/cf_mappings.pkl
✅ cf_train.csv                        /kaggle/input/datasets/trnquangtriu/7dataset/cf_train.csv
✅ cf_config.json                      /kaggle/input/datasets/trnquangtriu/7dataset/cf_config

In [5]:
%pip install -q implicit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 13.7 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [6]:
# ============================================================
# STEP 7 - CELL 3
# LOAD DATA AND CF MODEL
# ============================================================

assert not missing_required, (
    "Please add missing datasets before continuing."
)


print("=" * 70)
print("LOAD STEP 7 DATA")
print("=" * 70)


# ------------------------------------------------------------
# 1. Product catalog
# ------------------------------------------------------------

products = pd.read_csv(
    artifact_paths["products.csv"],
    low_memory=False
)


# ------------------------------------------------------------
# 2. Interactions
# ------------------------------------------------------------

interactions = pd.read_csv(
    artifact_paths["interactions.csv"],
    low_memory=False
)

cf_interactions = pd.read_csv(
    artifact_paths["cf_interactions.csv"],
    low_memory=False
)


# ------------------------------------------------------------
# 3. CF training history
# ------------------------------------------------------------

cf_train = pd.read_csv(
    artifact_paths["cf_train.csv"],
    low_memory=False
)


# ------------------------------------------------------------
# 4. Product embeddings
# ------------------------------------------------------------

product_embeddings = np.load(
    artifact_paths["product_embeddings.npy"],
    mmap_mode="r"
)


# ------------------------------------------------------------
# 5. BPR model
# ------------------------------------------------------------

with open(
    artifact_paths["bpr_model.pkl"],
    "rb"
) as f:

    bpr_model = pickle.load(f)


# ------------------------------------------------------------
# 6. CF mappings
# ------------------------------------------------------------

with open(
    artifact_paths["cf_mappings.pkl"],
    "rb"
) as f:

    cf_mappings = pickle.load(f)


user_to_idx = cf_mappings["user_to_idx"]

item_to_idx = cf_mappings["item_to_idx"]

idx_to_user = cf_mappings["idx_to_user"]

idx_to_item = cf_mappings["idx_to_item"]


# ------------------------------------------------------------
# 7. CF configuration
# ------------------------------------------------------------

with open(
    artifact_paths["cf_config.json"],
    "r",
    encoding="utf-8"
) as f:

    cf_config = json.load(f)


# ------------------------------------------------------------
# 8. Summary
# ------------------------------------------------------------

print("\nPRODUCT CATALOG")
print("Rows:", f"{len(products):,}")
print("Columns:", products.columns.tolist())


print("\nINTERACTIONS")
print("Rows:", f"{len(interactions):,}")


print("\nCF CORE")
print("Rows:", f"{len(cf_interactions):,}")


print("\nCF TRAIN")
print("Rows:", f"{len(cf_train):,}")
print("Users:", f"{len(user_to_idx):,}")
print("Products:", f"{len(item_to_idx):,}")


print("\nPRODUCT EMBEDDINGS")
print("Shape:", product_embeddings.shape)
print("Dtype:", product_embeddings.dtype)


print("\nSELECTED CF MODEL")
print(
    cf_config.get(
        "selected_model",
        "UNKNOWN"
    )
)


print("\n✅ Step 7 data loaded")

LOAD STEP 7 DATA

PRODUCT CATALOG
Rows: 164,926
Columns: ['product_id', 'product_name', 'category', 'subcategory', 'brand', 'price', 'currency', 'average_rating', 'review_count', 'sold_count', 'description', 'image_url', 'product_url', 'source', 'language']

INTERACTIONS
Rows: 767,179

CF CORE
Rows: 259,427

CF TRAIN
Rows: 205,621
Users: 26,903
Products: 15,546

PRODUCT EMBEDDINGS
Shape: (164926, 384)
Dtype: float32

SELECTED CF MODEL
BPR_F64_R05

✅ Step 7 data loaded


In [7]:
# ============================================================
# STEP 7 - CELL 4
# PRODUCT EMBEDDING ALIGNMENT AUDIT
# ============================================================

print("=" * 70)
print("PRODUCT EMBEDDING ALIGNMENT AUDIT")
print("=" * 70)


# ------------------------------------------------------------
# 1. Product columns
# ------------------------------------------------------------

print("\n[1] PRODUCT COLUMNS")

for col in products.columns:

    print(col)


# ------------------------------------------------------------
# 2. Embedding metadata candidates
# ------------------------------------------------------------

print("\n[2] EMBEDDING METADATA")

metadata_candidates = [
    "product_embedding_metadata.csv",
    "product_metadata.csv",
    "semantic_products.csv",
    "clean_products.csv"
]

for filename in metadata_candidates:

    path = artifact_paths.get(
        filename
    )

    if path is None:
        continue

    print("\nFile:", filename)

    metadata_preview = pd.read_csv(
        path,
        nrows=5
    )

    print(
        "Columns:",
        metadata_preview.columns.tolist()
    )

    display(
        metadata_preview.head(3)
    )


# ------------------------------------------------------------
# 3. Product ID samples
# ------------------------------------------------------------

print("\n[3] PRODUCT ID SAMPLE")

display(
    products[
        [
            c
            for c in [
                "product_id",
                "source",
                "product_name"
            ]
            if c in products.columns
        ]
    ].head(10)
)


# ------------------------------------------------------------
# 4. Embedding shape
# ------------------------------------------------------------

print("\n[4] EMBEDDING SHAPE")

print(
    "Product rows:",
    f"{len(products):,}"
)

print(
    "Embedding rows:",
    f"{product_embeddings.shape[0]:,}"
)

print(
    "Dimension:",
    product_embeddings.shape[1]
)


# ------------------------------------------------------------
# 5. CF mapping audit
# ------------------------------------------------------------

print("\n[5] CF MAPPING AUDIT")

print(
    "Users:",
    len(user_to_idx)
)

print(
    "Items:",
    len(item_to_idx)
)

print(
    "User factors:",
    bpr_model.user_factors.shape
)

print(
    "Item factors:",
    bpr_model.item_factors.shape
)


assert (
    bpr_model.user_factors.shape[0]
    == len(user_to_idx)
)

assert (
    bpr_model.item_factors.shape[0]
    == len(item_to_idx)
)

assert (
    product_embeddings.shape[0]
    == len(products)
)

print(
    "\n✅ Shape checks passed"
)

print(
    "⚠ Product ID alignment must still be verified."
)

PRODUCT EMBEDDING ALIGNMENT AUDIT

[1] PRODUCT COLUMNS
product_id
product_name
category
subcategory
brand
price
currency
average_rating
review_count
sold_count
description
image_url
product_url
source
language

[2] EMBEDDING METADATA

File: product_embedding_metadata.csv
Columns: ['product_id', 'product_name', 'category', 'subcategory', 'brand', 'price', 'currency', 'average_rating', 'review_count', 'sold_count', 'description', 'image_url', 'product_url', 'source', 'language', 'clean_product_name', 'clean_category', 'clean_subcategory', 'clean_brand', 'clean_description', 'product_text', 'product_text_word_count', 'semantic_usable', 'product_embedding_id']


,product_id,product_name,category,subcategory,brand,price,currency,average_rating,review_count,sold_count,description,image_url,product_url,source,language,clean_product_name,clean_category,clean_subcategory,clean_brand,clean_description,product_text,product_text_word_count,semantic_usable,product_embedding_id
0,amazon_B07PXGQC1Q,Apple AirPods (2nd Generation) Wireless Earbuds with Lightning Charging Case Included. Over 24 H...,Apple Products,NaN,Apple,99.00,USD,4.7,585624,NaN,NaN,NaN,NaN,amazon,en,Apple AirPods (2nd Generation) Wireless Earbuds with Lightning Charging Case Included. Over 24 H...,Apple Products,NaN,Apple,NaN,Product: Apple AirPods (2nd Generation) Wireless Earbuds with Lightning Charging Case Included. ...,31,True,0
1,amazon_B07KTYJ769,"Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Device",Amazon Devices,NaN,Amazon,24.99,USD,4.7,542825,NaN,NaN,NaN,NaN,amazon,en,"Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Device",Amazon Devices,NaN,Amazon,NaN,"Product: Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Devic...",23,True,1
2,amazon_B0BGNG1294,"Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, Black",Home Audio & Theater,NaN,Amazon Basics,8.55,USD,4.7,507202,NaN,NaN,NaN,NaN,amazon,en,"Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, Black",Home Audio & Theater,NaN,Amazon Basics,NaN,"Product: Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, B...",24,True,2



File: clean_products.csv
Columns: ['product_id', 'product_name', 'category', 'subcategory', 'brand', 'price', 'currency', 'average_rating', 'review_count', 'sold_count', 'description', 'image_url', 'product_url', 'source', 'language', 'clean_product_name', 'clean_category', 'clean_subcategory', 'clean_brand', 'clean_description', 'product_text', 'product_text_word_count', 'semantic_usable']


,product_id,product_name,category,subcategory,brand,price,currency,average_rating,review_count,sold_count,description,image_url,product_url,source,language,clean_product_name,clean_category,clean_subcategory,clean_brand,clean_description,product_text,product_text_word_count,semantic_usable
0,amazon_B07PXGQC1Q,Apple AirPods (2nd Generation) Wireless Earbuds with Lightning Charging Case Included. Over 24 H...,Apple Products,NaN,Apple,99.00,USD,4.7,585624,NaN,NaN,NaN,NaN,amazon,en,Apple AirPods (2nd Generation) Wireless Earbuds with Lightning Charging Case Included. Over 24 H...,Apple Products,NaN,Apple,NaN,Product: Apple AirPods (2nd Generation) Wireless Earbuds with Lightning Charging Case Included. ...,31,True
1,amazon_B07KTYJ769,"Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Device",Amazon Devices,NaN,Amazon,24.99,USD,4.7,542825,NaN,NaN,NaN,NaN,amazon,en,"Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Device",Amazon Devices,NaN,Amazon,NaN,"Product: Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Devic...",23,True
2,amazon_B0BGNG1294,"Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, Black",Home Audio & Theater,NaN,Amazon Basics,8.55,USD,4.7,507202,NaN,NaN,NaN,NaN,amazon,en,"Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, Black",Home Audio & Theater,NaN,Amazon Basics,NaN,"Product: Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, B...",24,True



[3] PRODUCT ID SAMPLE


,product_id,source,product_name
0,amazon_B07PXGQC1Q,amazon,Apple AirPods (2nd Generation) Wireless Earbuds with Lightning Charging Case Included. Over 24 H...
1,amazon_B07KTYJ769,amazon,"Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Device"
2,amazon_B0BGNG1294,amazon,"Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, Black"
3,amazon_B08CLNX58K,amazon,SanDisk 32GB 2-Pack Ultra MicroSDHC UHS-I Memory Card (2x32GB) - SDSQUAR-032G-GN6MT
4,amazon_B08WJSHSLC,amazon,Fire TV Stick (3rd Gen) with Alexa Voice Remote (includes TV controls) + Star Wars The Mandalori...
5,amazon_B07HZLHPKP,amazon,"Echo Show 5 (1st Gen, 2019 release) -- Smart display with Alexa – stay connected with video call..."
6,amazon_B08XPWDSWW,amazon,TOZO T10 Bluetooth 5.3 Wireless Earbuds with Wireless Charging Case IPX8 Waterproof Stereo Headp...
7,amazon_B09V1FT19S,amazon,"SanDisk 256GB Extreme microSDXC UHS-I Memory Card with Adapter - Up to 160MB/s, C10, U3, V30, 4K..."
8,amazon_B08F9ZCTCL,amazon,"Fire TV Stick Lite, free and live TV, Alexa Voice Remote Lite, smart home controls, HD streaming"
9,amazon_B0BSFT117F,amazon,SanDisk Cruzer Blade 16GB USB 2.0 Flash Drive



[4] EMBEDDING SHAPE
Product rows: 164,926
Embedding rows: 164,926
Dimension: 384

[5] CF MAPPING AUDIT
Users: 26903
Items: 15546
User factors: (26903, 65)
Item factors: (15546, 65)

✅ Shape checks passed
⚠ Product ID alignment must still be verified.


# Xác nhận Product Embedding Alignment

In [8]:
# ============================================================
# STEP 7 - CELL 5
# VERIFY PRODUCT EMBEDDING ALIGNMENT
# ============================================================

print("=" * 70)
print("PRODUCT EMBEDDING ALIGNMENT VERIFICATION")
print("=" * 70)

embedding_metadata = pd.read_csv(
    artifact_paths["product_embedding_metadata.csv"],
    low_memory=False
)

# ------------------------------------------------------------
# 1. Basic checks
# ------------------------------------------------------------

checks = {}

checks["metadata_rows"] = (
    len(embedding_metadata)
    == product_embeddings.shape[0]
)

checks["product_rows"] = (
    len(products)
    == len(embedding_metadata)
)

checks["product_ids_unique"] = (
    embedding_metadata["product_id"].is_unique
)

checks["embedding_ids_unique"] = (
    embedding_metadata["product_embedding_id"].is_unique
)

checks["embedding_ids_complete"] = (
    set(
        embedding_metadata["product_embedding_id"]
    )
    == set(
        range(len(product_embeddings))
    )
)

# ------------------------------------------------------------
# 2. Verify product order
# ------------------------------------------------------------

checks["product_order_matches"] = (
    products["product_id"]
    .astype(str)
    .reset_index(drop=True)
    .equals(
        embedding_metadata["product_id"]
        .astype(str)
        .reset_index(drop=True)
    )
)

# ------------------------------------------------------------
# 3. Verify embedding IDs match row positions
# ------------------------------------------------------------

checks["embedding_order_matches"] = (
    np.array_equal(
        embedding_metadata[
            "product_embedding_id"
        ].to_numpy(),

        np.arange(
            len(embedding_metadata)
        )
    )
)

# ------------------------------------------------------------
# 4. Display checks
# ------------------------------------------------------------

for key, value in checks.items():

    icon = "✅" if value else "❌"

    print(
        f"{icon} {key:<30}: {value}"
    )

assert all(checks.values()), (
    "Embedding alignment failed. "
    "Do not continue until mapping is corrected."
)

# ------------------------------------------------------------
# 5. Build product mappings
# ------------------------------------------------------------

product_to_embedding_idx = dict(
    zip(
        embedding_metadata["product_id"],
        embedding_metadata["product_embedding_id"]
    )
)

embedding_idx_to_product = (
    embedding_metadata["product_id"]
    .to_numpy()
)

print("\nProduct mappings:",
      f"{len(product_to_embedding_idx):,}")

print("\nSample mappings:")

display(
    embedding_metadata[
        [
            "product_id",
            "product_embedding_id"
        ]
    ].head(10)
)

print(
    "\n✅ Product embedding alignment verified"
)

PRODUCT EMBEDDING ALIGNMENT VERIFICATION
✅ metadata_rows                 : True
✅ product_rows                  : True
✅ product_ids_unique            : True
✅ embedding_ids_unique          : True
✅ embedding_ids_complete        : True
✅ product_order_matches         : True
✅ embedding_order_matches       : True

Product mappings: 164,926

Sample mappings:


,product_id,product_embedding_id
0,amazon_B07PXGQC1Q,0
1,amazon_B07KTYJ769,1
2,amazon_B0BGNG1294,2
3,amazon_B08CLNX58K,3
4,amazon_B08WJSHSLC,4
5,amazon_B07HZLHPKP,5
6,amazon_B08XPWDSWW,6
7,amazon_B09V1FT19S,7
8,amazon_B08F9ZCTCL,8
9,amazon_B0BSFT117F,9



✅ Product embedding alignment verified


# Kiểm tra lịch sử Train và khả năng tạo Semantic User Profile

In [9]:
# ============================================================
# STEP 7 - CELL 6
# TRAIN HISTORY SEMANTIC COVERAGE
# ============================================================

print("=" * 70)
print("TRAIN HISTORY SEMANTIC COVERAGE")
print("=" * 70)

cf_train = cf_train.copy()

cf_train["embedding_idx"] = (
    cf_train["product_id"]
    .map(product_to_embedding_idx)
)

valid_history = (
    cf_train["embedding_idx"]
    .notna()
)

print(
    "Train interactions:",
    f"{len(cf_train):,}"
)

print(
    "Interactions with embeddings:",
    f"{valid_history.sum():,}"
)

print(
    "Embedding coverage:",
    f"{valid_history.mean()*100:.2f}%"
)

print(
    "\nUsers with semantic history:",
    f"{cf_train.loc[valid_history, 'user_id'].nunique():,}"
)

print(
    "Users in CF mapping:",
    f"{len(user_to_idx):,}"
)

missing_products = (
    cf_train.loc[
        ~valid_history,
        "product_id"
    ]
    .unique()
)

print(
    "\nMissing product embeddings:",
    len(missing_products)
)

assert valid_history.all(), (
    "Some CF train products have no embeddings."
)

print(
    "\n✅ All CF train interactions have product embeddings"
)

TRAIN HISTORY SEMANTIC COVERAGE
Train interactions: 205,621
Interactions with embeddings: 205,621
Embedding coverage: 100.00%

Users with semantic history: 26,903
Users in CF mapping: 26,903

Missing product embeddings: 0

✅ All CF train interactions have product embeddings


# Tái lập temporal split từ Train đã lưu

In [10]:
# ============================================================
# STEP 7 - CELL 7
# RECONSTRUCT TEMPORAL SPLIT FROM SAVED CF TRAIN
# ============================================================

print("=" * 70)
print("RECONSTRUCT TEMPORAL SPLIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Prepare original CF core
# ------------------------------------------------------------

core = cf_interactions.copy()

saved_train = cf_train.copy()

KEY_COLS = [
    "user_id",
    "product_id",
    "rating",
    "timestamp"
]

for col in KEY_COLS:

    assert col in core.columns, f"Missing in core: {col}"
    assert col in saved_train.columns, f"Missing in train: {col}"

for df in [core, saved_train]:

    df["user_id"] = df["user_id"].astype(str)

    df["product_id"] = df["product_id"].astype(str)

    df["rating"] = pd.to_numeric(
        df["rating"],
        errors="raise"
    ).round(6)

    df["timestamp"] = pd.to_numeric(
        df["timestamp"],
        errors="raise"
    ).astype("int64")


# ------------------------------------------------------------
# 2. Handle repeated identical records safely
# ------------------------------------------------------------

core["_occurrence"] = (
    core.groupby(
        KEY_COLS,
        dropna=False
    ).cumcount()
)

saved_train["_occurrence"] = (
    saved_train.groupby(
        KEY_COLS,
        dropna=False
    ).cumcount()
)

MATCH_COLS = KEY_COLS + ["_occurrence"]


# ------------------------------------------------------------
# 3. Mark records belonging to saved Train
# ------------------------------------------------------------

train_keys = (
    saved_train[MATCH_COLS]
    .copy()
)

train_keys["_in_saved_train"] = True

reconstructed = core.merge(
    train_keys,
    on=MATCH_COLS,
    how="left",
    validate="one_to_one"
)

train_mask = (
    reconstructed["_in_saved_train"]
    .fillna(False)
    .astype(bool)
)

reconstructed_train = (
    reconstructed.loc[train_mask]
    .copy()
)

holdout = (
    reconstructed.loc[~train_mask]
    .copy()
)


# ------------------------------------------------------------
# 4. Verify exact Train reconstruction
# ------------------------------------------------------------

print("\n[1] TRAIN RECONSTRUCTION")

print(
    "Saved train       :",
    f"{len(saved_train):,}"
)

print(
    "Reconstructed     :",
    f"{len(reconstructed_train):,}"
)

assert len(reconstructed_train) == len(saved_train), (
    "Saved Train does not match CF core."
)

assert len(core) == (
    len(reconstructed_train) + len(holdout)
)

print("✅ Saved Train recovered")


# ------------------------------------------------------------
# 5. Verify two holdout records per user
# ------------------------------------------------------------

holdout_counts = (
    holdout.groupby("user_id")
    .size()
)

print("\n[2] HOLDOUT CHECK")

print(
    "Holdout interactions:",
    f"{len(holdout):,}"
)

print(
    "Holdout users:",
    f"{holdout['user_id'].nunique():,}"
)

print(
    "Holdout per user distribution:"
)

print(
    holdout_counts.value_counts()
    .sort_index()
)

assert (
    len(holdout)
    == 2 * len(user_to_idx)
), "Expected Validation 1 + Test 1 per user."

assert (
    holdout_counts.eq(2).all()
), "Not every user has exactly two holdout records."


# ------------------------------------------------------------
# 6. Reconstruct chronological Validation and Test
# ------------------------------------------------------------

holdout = (
    holdout.sort_values(
        [
            "user_id",
            "timestamp",
            "product_id"
        ],
        kind="mergesort"
    )
    .copy()
)

holdout["_holdout_order"] = (
    holdout.groupby("user_id")
    .cumcount()
)

validation = (
    holdout[
        holdout["_holdout_order"] == 0
    ]
    .copy()
)

test = (
    holdout[
        holdout["_holdout_order"] == 1
    ]
    .copy()
)


# ------------------------------------------------------------
# 7. Remove temporary columns
# ------------------------------------------------------------

temporary_cols = [
    "_occurrence",
    "_in_saved_train",
    "_holdout_order"
]

train = (
    reconstructed_train
    .drop(
        columns=temporary_cols,
        errors="ignore"
    )
    .reset_index(drop=True)
)

validation = (
    validation
    .drop(
        columns=temporary_cols,
        errors="ignore"
    )
    .reset_index(drop=True)
)

test = (
    test
    .drop(
        columns=temporary_cols,
        errors="ignore"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RECONSTRUCTED SPLIT")
print("=" * 70)

print(
    "Train      :",
    f"{len(train):,}"
)

print(
    "Validation :",
    f"{len(validation):,}"
)

print(
    "Test       :",
    f"{len(test):,}"
)

print(
    "Total      :",
    f"{len(train)+len(validation)+len(test):,}"
)

assert len(train) == 205621
assert len(validation) == 26903
assert len(test) == 26903

print("\n✅ Temporal split reconstructed")

RECONSTRUCT TEMPORAL SPLIT

[1] TRAIN RECONSTRUCTION
Saved train       : 205,621
Reconstructed     : 205,621
✅ Saved Train recovered

[2] HOLDOUT CHECK
Holdout interactions: 53,806
Holdout users: 26,903
Holdout per user distribution:
2    26903
Name: count, dtype: int64

RECONSTRUCTED SPLIT
Train      : 205,621
Validation : 26,903
Test       : 26,903
Total      : 259,427

✅ Temporal split reconstructed


# Kiểm tra temporal leakage

In [11]:
# ============================================================
# STEP 7 - CELL 8
# TEMPORAL LEAKAGE AUDIT
# ============================================================

print("=" * 70)
print("TEMPORAL LEAKAGE AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Train maximum timestamp per user
# ------------------------------------------------------------

train_max_time = (
    train.groupby("user_id")["timestamp"]
    .max()
    .rename("train_max_timestamp")
)


# ------------------------------------------------------------
# 2. Validation and Test timestamps
# ------------------------------------------------------------

val_time = (
    validation.set_index("user_id")[
        "timestamp"
    ]
    .rename("validation_timestamp")
)

test_time = (
    test.set_index("user_id")[
        "timestamp"
    ]
    .rename("test_timestamp")
)


# ------------------------------------------------------------
# 3. Combine
# ------------------------------------------------------------

temporal_audit = pd.concat(
    [
        train_max_time,
        val_time,
        test_time
    ],
    axis=1
)


# ------------------------------------------------------------
# 4. Check chronology
# ------------------------------------------------------------

temporal_audit["valid_order"] = (
    (
        temporal_audit["train_max_timestamp"]
        <= temporal_audit["validation_timestamp"]
    )
    &
    (
        temporal_audit["validation_timestamp"]
        <= temporal_audit["test_timestamp"]
    )
)

temporal_audit["holdout_timestamp_tie"] = (
    temporal_audit["validation_timestamp"]
    == temporal_audit["test_timestamp"]
)

temporal_audit["train_validation_tie"] = (
    temporal_audit["train_max_timestamp"]
    == temporal_audit["validation_timestamp"]
)


# ------------------------------------------------------------
# 5. Results
# ------------------------------------------------------------

print(
    "Users:",
    f"{len(temporal_audit):,}"
)

print(
    "Valid chronological order:",
    f"{temporal_audit['valid_order'].sum():,}"
)

print(
    "Invalid chronological order:",
    f"{(~temporal_audit['valid_order']).sum():,}"
)

print(
    "Validation/Test timestamp ties:",
    f"{temporal_audit['holdout_timestamp_tie'].sum():,}"
)

print(
    "Train/Validation timestamp ties:",
    f"{temporal_audit['train_validation_tie'].sum():,}"
)

assert temporal_audit[
    "valid_order"
].all(), (
    "Temporal leakage detected."
)

print("\n[1] SAMPLE")

display(
    temporal_audit.head()
)

print("\n[2] KNOWN-ITEM COVERAGE")

for name, df in [
    ("Validation", validation),
    ("Test", test)
]:

    known = (
        df["product_id"]
        .isin(item_to_idx)
    )

    print(
        f"{name:<12}: "
        f"{known.sum():,}/{len(df):,} "
        f"({known.mean()*100:.2f}%)"
    )

print(
    "\n✅ Temporal chronology checks passed"
)

TEMPORAL LEAKAGE AUDIT
Users: 26,903
Valid chronological order: 26,903
Invalid chronological order: 0
Validation/Test timestamp ties: 0
Train/Validation timestamp ties: 0

[1] SAMPLE


,train_max_timestamp,validation_timestamp,test_timestamp,valid_order,holdout_timestamp_tie,train_validation_tie
user_id,,,,,,
amazon_AE22DOLTN2WN6B3TJ32UDXOEFNSA,1462826055000,1487812619000,1503463342573,True,False,False
amazon_AE22MRZKBD5MWJLIMGYLLJ67ZWVQ,1481195698000,1490613235000,1535123305276,True,False,False
amazon_AE22PJ54OVIRX3I6KSLMPRHPHA4A,1614475712590,1621368480432,1638930544981,True,False,False
amazon_AE22U2DM2E4EOEDEEZ46BDVIYPOQ,1650565841774,1660665772804,1667750100274,True,False,False
amazon_AE22Z3RLVIRU6RT5PNRK5CFFNEFQ,1605235204157,1616866936646,1642953499250,True,False,False



[2] KNOWN-ITEM COVERAGE
Validation  : 26,745/26,903 (99.41%)
Test        : 26,585/26,903 (98.82%)

✅ Temporal chronology checks passed


# Tạo User Semantic Profile

In [12]:
# ============================================================
# STEP 7 - CELL 9
# BUILD USER SEMANTIC PROFILES
# ============================================================

print("=" * 70)
print("BUILD USER SEMANTIC PROFILES")
print("=" * 70)

# ------------------------------------------------------------
# 1. Prepare training history
# ------------------------------------------------------------

semantic_train = train[
    [
        "user_id",
        "product_id",
        "rating"
    ]
].copy()

semantic_train["embedding_idx"] = (
    semantic_train["product_id"]
    .map(product_to_embedding_idx)
)

assert semantic_train[
    "embedding_idx"
].notna().all()


# ------------------------------------------------------------
# 2. Identify users with positive history
# ------------------------------------------------------------

semantic_train["is_positive"] = (
    semantic_train["rating"] >= 4
)

has_positive = (
    semantic_train.groupby("user_id")[
        "is_positive"
    ]
    .transform("any")
)


# ------------------------------------------------------------
# 3. Positive preference weights
# ------------------------------------------------------------

semantic_train["semantic_weight"] = np.where(
    has_positive,

    np.where(
        semantic_train["is_positive"],
        semantic_train["rating"] / 5.0,
        0.0
    ),

    1.0
)


# ------------------------------------------------------------
# 4. Build sparse user-item weight matrix
# ------------------------------------------------------------

semantic_train["user_idx"] = (
    semantic_train["user_id"]
    .map(user_to_idx)
)

user_indices = (
    semantic_train["user_idx"]
    .astype(int)
    .to_numpy()
)

embedding_indices = (
    semantic_train["embedding_idx"]
    .astype(int)
    .to_numpy()
)

weights = (
    semantic_train["semantic_weight"]
    .astype(np.float32)
    .to_numpy()
)


user_semantic_matrix = csr_matrix(
    (
        weights,
        (
            user_indices,
            embedding_indices
        )
    ),

    shape=(
        len(user_to_idx),
        len(product_embeddings)
    ),

    dtype=np.float32
)


# ------------------------------------------------------------
# 5. Compute user profiles
# ------------------------------------------------------------

print("\nComputing user semantic vectors...")

user_semantic_profiles = (
    user_semantic_matrix
    @ np.asarray(
        product_embeddings,
        dtype=np.float32
    )
)

user_semantic_profiles = np.asarray(
    user_semantic_profiles,
    dtype=np.float32
)


# ------------------------------------------------------------
# 6. L2 normalize
# ------------------------------------------------------------

profile_norms = np.linalg.norm(
    user_semantic_profiles,
    axis=1,
    keepdims=True
)

user_semantic_profiles = (
    user_semantic_profiles
    /
    np.maximum(
        profile_norms,
        1e-12
    )
)


# ------------------------------------------------------------
# 7. Audit
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("USER SEMANTIC PROFILE SUMMARY")
print("=" * 70)

print(
    "Users:",
    f"{user_semantic_profiles.shape[0]:,}"
)

print(
    "Dimension:",
    user_semantic_profiles.shape[1]
)

print(
    "Shape:",
    user_semantic_profiles.shape
)

print(
    "Users with positive history:",
    f"{semantic_train.loc[has_positive, 'user_id'].nunique():,}"
)

fallback_users = (
    semantic_train.loc[
        ~has_positive,
        "user_id"
    ].nunique()
)

print(
    "Fallback users:",
    f"{fallback_users:,}"
)

zero_profiles = (
    np.linalg.norm(
        user_semantic_profiles,
        axis=1
    ) < 1e-8
).sum()

print(
    "Zero profiles:",
    int(zero_profiles)
)

assert (
    user_semantic_profiles.shape
    ==
    (
        len(user_to_idx),
        product_embeddings.shape[1]
    )
)

assert np.isfinite(
    user_semantic_profiles
).all()

assert zero_profiles == 0

print(
    "\n✅ User semantic profiles created successfully"
)

BUILD USER SEMANTIC PROFILES

Computing user semantic vectors...

USER SEMANTIC PROFILE SUMMARY
Users: 26,903
Dimension: 384
Shape: (26903, 384)
Users with positive history: 26,641
Fallback users: 262
Zero profiles: 0

✅ User semantic profiles created successfully


# CHUẨN BỊ HYBRID RANKING

In [13]:
# ============================================================
# STEP 7 - CELL 10
# PREPARE PRODUCT CATALOG AND TRAIN-ONLY POPULARITY
# ============================================================

print("=" * 70)
print("PREPARE PRODUCT CATALOG AND POPULARITY")
print("=" * 70)

# ------------------------------------------------------------
# 1. Product catalog
# ------------------------------------------------------------

product_catalog = products.copy()

assert product_catalog["product_id"].is_unique

product_catalog["embedding_idx"] = (
    product_catalog["product_id"]
    .map(product_to_embedding_idx)
)

assert product_catalog["embedding_idx"].notna().all()

product_catalog["embedding_idx"] = (
    product_catalog["embedding_idx"].astype(int)
)

# ------------------------------------------------------------
# 2. Train-only popularity
# ------------------------------------------------------------

train_popularity = (
    train.groupby("product_id")
    .agg(
        train_interactions=("user_id", "size"),
        train_avg_rating=("rating", "mean")
    )
    .reset_index()
)

train_popularity["popularity_score"] = (
    np.log1p(
        train_popularity["train_interactions"]
    )
)

max_popularity = (
    train_popularity["popularity_score"].max()
)

train_popularity["popularity_score"] /= (
    max_popularity
    if max_popularity > 0
    else 1.0
)

# ------------------------------------------------------------
# 3. Merge with catalog
# ------------------------------------------------------------

product_catalog = product_catalog.merge(
    train_popularity,
    on="product_id",
    how="left",
    validate="one_to_one"
)

product_catalog["train_interactions"] = (
    product_catalog["train_interactions"]
    .fillna(0)
    .astype(int)
)

product_catalog["popularity_score"] = (
    product_catalog["popularity_score"]
    .fillna(0.0)
    .astype(np.float32)
)

# ------------------------------------------------------------
# 4. Build fast lookup arrays
# ------------------------------------------------------------

catalog_by_embedding = (
    product_catalog
    .sort_values("embedding_idx")
    .reset_index(drop=True)
)

assert np.array_equal(
    catalog_by_embedding["embedding_idx"].to_numpy(),
    np.arange(len(product_embeddings))
)

embedding_product_ids = (
    catalog_by_embedding["product_id"]
    .astype(str)
    .to_numpy()
)

embedding_sources = (
    catalog_by_embedding["source"]
    .astype(str)
    .to_numpy()
)

embedding_popularity = (
    catalog_by_embedding["popularity_score"]
    .to_numpy(dtype=np.float32)
)

# ------------------------------------------------------------
# 5. Amazon evaluation catalog
# ------------------------------------------------------------

amazon_embedding_indices = np.flatnonzero(
    embedding_sources == "amazon"
)

amazon_product_ids = (
    embedding_product_ids[
        amazon_embedding_indices
    ]
)

# ------------------------------------------------------------
# 6. Popularity ranking
# ------------------------------------------------------------

amazon_popularity_order = (
    amazon_embedding_indices[
        np.argsort(
            -embedding_popularity[
                amazon_embedding_indices
            ],
            kind="stable"
        )
    ]
)

# ------------------------------------------------------------
# 7. Summary
# ------------------------------------------------------------

print("\nFull catalog:",
      f"{len(product_catalog):,}")

print("Amazon products:",
      f"{len(amazon_embedding_indices):,}")

print("Products with Train interactions:",
      f"{(product_catalog['train_interactions'] > 0).sum():,}")

print("\nPopularity statistics:")

display(
    product_catalog[
        product_catalog["train_interactions"] > 0
    ]["popularity_score"].describe()
)

print("\nTop Amazon products by Train popularity:")

display(
    catalog_by_embedding.iloc[
        amazon_popularity_order[:10]
    ][
        [
            "product_id",
            "product_name",
            "train_interactions",
            "popularity_score"
        ]
    ]
)

print("\n✅ Train-only popularity ready")

PREPARE PRODUCT CATALOG AND POPULARITY

Full catalog: 164,926
Amazon products: 121,917
Products with Train interactions: 15,546

Popularity statistics:


count    15546.000000
mean         0.306556
std          0.107288
min          0.095777
25%          0.247581
50%          0.287332
75%          0.354419
max          1.000000
Name: popularity_score, dtype: float64


Top Amazon products by Train popularity:


,product_id,product_name,train_interactions,popularity_score
14,amazon_B01K8B8YA8,Echo Dot (2nd Generation) - Smart speaker with Alexa - White,1389,1.000000
38,amazon_B07S764D9V,"Panasonic ErgoFit Wired Earbuds, In-Ear Headphones with Microphone and Call Controller, Ergonomi...",938,0.945801
60,amazon_B011BRUOMO,"SanDisk Ultra 32GB microSDHC UHS-I Card with Adapter, Silver, Standard Packaging (SDSQUNC-032G-G...",805,0.924697
2,amazon_B0BGNG1294,"Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, Black",623,0.889332
20,amazon_B0BW4PFM58,"OontZ Angle 3 Bluetooth Speaker, Portable Wireless Bluetooth 5.0 Speaker, 10 Watts, Crystal Clea...",527,0.866249
9362,amazon_B00X5C22SS,Sony BDPS3500 Blu-ray Player with Wi-Fi (2015 Model),469,0.850170
256,amazon_B07P9V8GSH,SanDisk Ultra 32GB UHS-I/Class 10 Micro SDHC Memory Card With Adapter - SDSDQUAN-032G-G4A,453,0.845384
1,amazon_B07KTYJ769,"Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Device",447,0.843546
307,amazon_B01G8JO5F2,"Senso Bluetooth Headphones, Best Wireless Sports Earbuds w/Mic IPX7 Waterproof HD Stereo Sweatpr...",418,0.834299
67,amazon_B00L0YLRUW,NETGEAR Wi-Fi Range Extender EX2700 - Coverage Up to 800 Sq Ft and 10 devices with N300 Wireless...,401,0.828576



✅ Train-only popularity ready


# Xây dựng User History và BPR Matrix

In [14]:
# ============================================================
# STEP 7 - CELL 11
# BUILD USER HISTORY AND BPR MATRIX
# ============================================================

print("=" * 70)
print("BUILD USER HISTORY AND BPR MATRIX")
print("=" * 70)

# ------------------------------------------------------------
# 1. User history
# ------------------------------------------------------------

user_history = (
    train.groupby("user_id")["product_id"]
    .agg(set)
    .to_dict()
)

user_history_count = (
    train.groupby("user_id")
    .size()
    .to_dict()
)

print(
    "Users with history:",
    f"{len(user_history):,}"
)

# ------------------------------------------------------------
# 2. Map Train to BPR indices
# ------------------------------------------------------------

bpr_train = train[
    [
        "user_id",
        "product_id",
        "rating"
    ]
].copy()

bpr_train["user_idx"] = (
    bpr_train["user_id"]
    .map(user_to_idx)
)

bpr_train["item_idx"] = (
    bpr_train["product_id"]
    .map(item_to_idx)
)

assert bpr_train["user_idx"].notna().all()
assert bpr_train["item_idx"].notna().all()

# ------------------------------------------------------------
# 3. Rebuild BPR user-item matrix
# ------------------------------------------------------------

# Match Step 6's rating / 5 weighting.
# BPR treats nonzero entries as observed interactions.

bpr_train["weight"] = (
    pd.to_numeric(
        bpr_train["rating"],
        errors="raise"
    )
    / 5.0
)

bpr_user_item_matrix = csr_matrix(
    (
        bpr_train["weight"].to_numpy(
            dtype=np.float32
        ),
        (
            bpr_train["user_idx"].astype(int),
            bpr_train["item_idx"].astype(int)
        )
    ),
    shape=(
        len(user_to_idx),
        len(item_to_idx)
    ),
    dtype=np.float32
)

# ------------------------------------------------------------
# 4. Audit
# ------------------------------------------------------------

print("\nBPR matrix:",
      bpr_user_item_matrix.shape)

print("Non-zero interactions:",
      f"{bpr_user_item_matrix.nnz:,}")

print("BPR user factors:",
      bpr_model.user_factors.shape)

print("BPR item factors:",
      bpr_model.item_factors.shape)

assert (
    bpr_user_item_matrix.shape[0]
    == bpr_model.user_factors.shape[0]
)

assert (
    bpr_user_item_matrix.shape[1]
    == bpr_model.item_factors.shape[0]
)

assert (
    len(user_history)
    == len(user_to_idx)
)

# ------------------------------------------------------------
# 5. History statistics
# ------------------------------------------------------------

history_counts = pd.Series(
    user_history_count
)

print("\nUser history statistics:")

display(
    history_counts.describe(
        percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]
    )
)

print("\n✅ User history and BPR matrix ready")

BUILD USER HISTORY AND BPR MATRIX
Users with history: 26,903

BPR matrix: (26903, 15546)
Non-zero interactions: 205,621
BPR user factors: (26903, 65)
BPR item factors: (15546, 65)

User history statistics:


count    26903.000000
mean         7.643051
std          8.012617
min          3.000000
25%          3.000000
50%          5.000000
75%          9.000000
90%         15.000000
95%         21.000000
max        238.000000
dtype: float64


✅ User history and BPR matrix ready


# Chuẩn bị FAISS Semantic Candidate Retrieval

In [16]:
%pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 49.6 MB/s eta 0:00:0000:0100:01m
Note: you may need to restart the kernel to use updated packages.


In [17]:
# ============================================================
# STEP 7 - CELL 12
# BUILD AMAZON SEMANTIC CANDIDATE INDEX
# ============================================================

import faiss

print("=" * 70)
print("BUILD AMAZON SEMANTIC INDEX")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load Amazon embeddings
# ------------------------------------------------------------

amazon_embeddings = np.ascontiguousarray(
    np.asarray(
        product_embeddings[
            amazon_embedding_indices
        ],
        dtype=np.float32
    )
)

# ------------------------------------------------------------
# 2. Normalize vectors
# ------------------------------------------------------------

faiss.normalize_L2(
    amazon_embeddings
)

# ------------------------------------------------------------
# 3. Build FAISS index
# ------------------------------------------------------------

embedding_dim = (
    amazon_embeddings.shape[1]
)

amazon_semantic_index = (
    faiss.IndexFlatIP(
        embedding_dim
    )
)

amazon_semantic_index.add(
    amazon_embeddings
)

# ------------------------------------------------------------
# 4. Check profile norms
# ------------------------------------------------------------

profile_norms = np.linalg.norm(
    user_semantic_profiles,
    axis=1
)

print("\nAmazon embeddings:",
      amazon_embeddings.shape)

print("FAISS vectors:",
      f"{amazon_semantic_index.ntotal:,}")

print("Embedding dimension:",
      amazon_semantic_index.d)

print("\nUser profile norm:")

print(
    "Minimum:",
    round(float(profile_norms.min()), 6)
)

print(
    "Maximum:",
    round(float(profile_norms.max()), 6)
)

# ------------------------------------------------------------
# 5. Test retrieval for one user
# ------------------------------------------------------------

sample_user_idx = 0

sample_profile = np.ascontiguousarray(
    user_semantic_profiles[
        sample_user_idx:
        sample_user_idx + 1
    ].astype(np.float32)
)

sample_scores, sample_indices = (
    amazon_semantic_index.search(
        sample_profile,
        10
    )
)

sample_embedding_indices = (
    amazon_embedding_indices[
        sample_indices[0]
    ]
)

sample_results = (
    catalog_by_embedding.iloc[
        sample_embedding_indices
    ][
        [
            "product_id",
            "product_name",
            "source",
            "price",
            "currency"
        ]
    ]
    .copy()
)

sample_results["semantic_score"] = (
    sample_scores[0]
)

print("\nSample semantic retrieval:")

display(
    sample_results.reset_index(
        drop=True
    )
)

# ------------------------------------------------------------
# 6. Final checks
# ------------------------------------------------------------

assert (
    amazon_semantic_index.ntotal
    == len(amazon_embedding_indices)
)

assert (
    amazon_semantic_index.d
    == user_semantic_profiles.shape[1]
)

print("\n✅ Amazon semantic index ready")

BUILD AMAZON SEMANTIC INDEX

Amazon embeddings: (121917, 384)
FAISS vectors: 121,917
Embedding dimension: 384

User profile norm:
Minimum: 1.0
Maximum: 1.0

Sample semantic retrieval:


,product_id,product_name,source,price,currency,semantic_score
0,amazon_B000I1GZ6O,"Stereo/VCR RCA Cable, 2 RCA (Audio) + RCA RG59 Video, Gold-Plated Connectors, 6 Foot",amazon,4.65,USD,0.800446
1,amazon_B0979Q7442,"FosPower RCA Y-Adapter (6 Feet), 1 RCA Male to 2 RCA Male Y Splitter Digital Stereo Audio Cable ...",amazon,9.99,USD,0.790129
2,amazon_B0BMPGB9QJ,"2RCA to 2RCA Cable 4 Feet, Hi-Fi Sound, Shielded, Gold-Plated, 2 RCA Male to 2 RCA Male Stereo A...",amazon,9.99,USD,0.783928
3,amazon_B00FMZWP7Y,Mediabridge™ Ultra Series Digital Audio Coaxial Cable (2 Feet) - Dual Shielded with RCA to RCA G...,amazon,7.84,USD,0.781924
4,amazon_B002VOAVA2,Focupix 25FT 5-RCA Component Video/Audio Coaxial Cable (RG-59/U),amazon,15.49,USD,0.777274
5,amazon_B08KHM6QJB,FosPower (6 Feet 2 RCA M/M Stereo Audio Cable [24K Gold Plated | Copper Core] 2RCA Male to 2RCA ...,amazon,9.99,USD,0.775627
6,amazon_B000F3Q7HM,Cables Unlimited AUD-1380-06 6 Feet RCA Component Video/Audio Cable,amazon,11.95,USD,0.774302
7,amazon_B0035494UU,RCA AH16250SN 16-Gauge 250 feet Speaker Wire,amazon,45.72,USD,0.772076
8,amazon_B0014KKLNQ,RCA PET20-7120 6 Feet Stereo Audio Cable,amazon,6.41,USD,0.767429
9,amazon_B019DIAVEO,"Stereo/VCR RCA Cable, 2 RCA (Audio) and RCA RG59 Video 25 Feet CNE63256",amazon,9.99,USD,0.767228



✅ Amazon semantic index ready


# Cấu hình Hybrid và chuẩn bị lookup

In [18]:
# ============================================================
# STEP 7 - CELL 13
# HYBRID CONFIGURATION AND LOOKUPS
# ============================================================

print("=" * 70)
print("HYBRID CONFIGURATION")
print("=" * 70)

SEMANTIC_CANDIDATES = 200
BPR_CANDIDATES = 200
POPULARITY_CANDIDATES = 200

MAX_K = 20

# Mapping embedding index -> Amazon FAISS index
embedding_to_amazon_idx = {
    int(emb_idx): int(local_idx)
    for local_idx, emb_idx in enumerate(
        amazon_embedding_indices
    )
}

# Mapping product ID -> catalog row
product_to_catalog_idx = {
    product_id: idx
    for idx, product_id in enumerate(
        embedding_product_ids
    )
}

# Mapping CF index -> product ID
def cf_idx_to_product_id(idx):

    idx = int(idx)

    if isinstance(idx_to_item, dict):
        return str(idx_to_item[idx])

    return str(idx_to_item[idx])


# Mapping product ID -> CF index
amazon_cf_product_ids = set(
    item_to_idx.keys()
)

# Normalized Amazon product embeddings
amazon_embeddings = np.ascontiguousarray(
    amazon_embeddings,
    dtype=np.float32
)

print("Semantic candidates :", SEMANTIC_CANDIDATES)
print("BPR candidates      :", BPR_CANDIDATES)
print("Popularity candidates:", POPULARITY_CANDIDATES)

print("\nAmazon catalog      :", f"{len(amazon_product_ids):,}")
print("Amazon CF products  :", f"{len(amazon_cf_product_ids):,}")

print("\n✅ Hybrid configuration ready")

HYBRID CONFIGURATION
Semantic candidates : 200
BPR candidates      : 200
Popularity candidates: 200

Amazon catalog      : 121,917
Amazon CF products  : 15,546

✅ Hybrid configuration ready


# Hàm lấy ứng viên từ Semantic, BPR và Popularity

In [19]:
# ============================================================
# STEP 7 - CELL 14
# HYBRID CANDIDATE GENERATION
# ============================================================

def get_semantic_candidates(
    user_id,
    n=200
):

    if user_id not in user_to_idx:
        return []

    user_idx = user_to_idx[user_id]

    profile = np.ascontiguousarray(
        user_semantic_profiles[
            user_idx:user_idx + 1
        ],
        dtype=np.float32
    )

    seen = user_history.get(
        user_id,
        set()
    )

    # Retrieve extra candidates to account for seen items
    search_n = min(
        amazon_semantic_index.ntotal,
        n + len(seen) + 20
    )

    scores, local_indices = (
        amazon_semantic_index.search(
            profile,
            search_n
        )
    )

    result = []

    for local_idx in local_indices[0]:

        if local_idx < 0:
            continue

        emb_idx = int(
            amazon_embedding_indices[
                local_idx
            ]
        )

        product_id = (
            embedding_product_ids[
                emb_idx
            ]
        )

        if product_id in seen:
            continue

        result.append(
            product_id
        )

        if len(result) >= n:
            break

    return result


def get_bpr_candidates(
    user_id,
    n=200
):

    if user_id not in user_to_idx:
        return []

    user_idx = user_to_idx[user_id]

    seen = user_history.get(
        user_id,
        set()
    )

    # Ask BPR for extra items so we can filter
    # the complete Train history consistently.
    request_n = min(
        len(item_to_idx),
        n + len(seen) + 20
    )

    ids, scores = bpr_model.recommend(
        userid=user_idx,
        user_items=bpr_user_item_matrix[
            user_idx
        ],
        N=request_n,
        filter_already_liked_items=True
    )

    result = []

    for idx in ids:

        product_id = (
            cf_idx_to_product_id(
                idx
            )
        )

        if product_id in seen:
            continue

        result.append(
            product_id
        )

        if len(result) >= n:
            break

    return result


def get_popularity_candidates(
    user_id,
    n=200
):

    seen = user_history.get(
        user_id,
        set()
    )

    result = []

    for emb_idx in amazon_popularity_order:

        product_id = (
            embedding_product_ids[
                emb_idx
            ]
        )

        if product_id in seen:
            continue

        result.append(
            product_id
        )

        if len(result) >= n:
            break

    return result


def generate_hybrid_candidates(
    user_id
):

    semantic_ids = get_semantic_candidates(
        user_id,
        SEMANTIC_CANDIDATES
    )

    bpr_ids = get_bpr_candidates(
        user_id,
        BPR_CANDIDATES
    )

    popularity_ids = get_popularity_candidates(
        user_id,
        POPULARITY_CANDIDATES
    )

    # Preserve order while removing duplicates
    candidate_ids = list(
        dict.fromkeys(
            semantic_ids
            + bpr_ids
            + popularity_ids
        )
    )

    return {
        "candidate_ids": candidate_ids,
        "semantic_ids": semantic_ids,
        "bpr_ids": bpr_ids,
        "popularity_ids": popularity_ids
    }


print("✅ Candidate generation functions ready")

✅ Candidate generation functions ready


In [20]:
# ============================================================
# STEP 7 - CELL 15
# COMPUTE HYBRID COMPONENT SCORES
# ============================================================

def compute_component_scores(
    user_id,
    candidate_ids
):

    if user_id not in user_to_idx:
        raise ValueError(
            f"Unknown CF user: {user_id}"
        )

    user_idx = user_to_idx[user_id]

    # --------------------------------------------------------
    # 1. Candidate embedding indices
    # --------------------------------------------------------

    candidate_emb_indices = np.array(
        [
            product_to_embedding_idx[pid]
            for pid in candidate_ids
        ],
        dtype=np.int64
    )

    candidate_vectors = np.asarray(
        product_embeddings[
            candidate_emb_indices
        ],
        dtype=np.float32
    )

    # --------------------------------------------------------
    # 2. Semantic scores
    # --------------------------------------------------------

    profile = (
        user_semantic_profiles[
            user_idx
        ]
    )

    semantic_cosine = (
        candidate_vectors @ profile
    )

    semantic_scores = np.clip(
        (semantic_cosine + 1.0) / 2.0,
        0.0,
        1.0
    )

    # --------------------------------------------------------
    # 3. BPR scores
    # --------------------------------------------------------

    user_factor = (
        bpr_model.user_factors[
            user_idx
        ]
    )

    all_bpr_raw = (
        bpr_model.item_factors
        @ user_factor
    )

    # Stable per-user normalization
    # over the full CF item catalog.
    bpr_min = float(
        all_bpr_raw.min()
    )

    bpr_max = float(
        all_bpr_raw.max()
    )

    bpr_range = max(
        bpr_max - bpr_min,
        1e-12
    )

    bpr_scores = np.zeros(
        len(candidate_ids),
        dtype=np.float32
    )

    cf_indices = np.array(
        [
            item_to_idx.get(
                pid,
                -1
            )
            for pid in candidate_ids
        ],
        dtype=np.int64
    )

    known_mask = (
        cf_indices >= 0
    )

    bpr_scores[
        known_mask
    ] = (
        all_bpr_raw[
            cf_indices[
                known_mask
            ]
        ]
        - bpr_min
    ) / bpr_range

    bpr_scores = np.clip(
        bpr_scores,
        0.0,
        1.0
    )

    # --------------------------------------------------------
    # 4. Popularity scores
    # --------------------------------------------------------

    popularity_scores = (
        embedding_popularity[
            candidate_emb_indices
        ]
    )

    # --------------------------------------------------------
    # 5. Build scoring DataFrame
    # --------------------------------------------------------

    result = pd.DataFrame({
        "product_id": candidate_ids,

        "semantic_score":
            semantic_scores,

        "bpr_score":
            bpr_scores,

        "popularity_score":
            popularity_scores,

        "in_cf":
            known_mask
    })

    return result


print("✅ Component scoring function ready")

✅ Component scoring function ready


# Hybrid Ranking và kiểm tra trên 5 user

In [21]:
# ============================================================
# STEP 7 - CELL 16
# HYBRID RANKING - SANITY TEST
# ============================================================

def recommend_hybrid(
    user_id,
    k=10,
    weights=None
):

    if weights is None:

        weights = {
            "semantic": 0.5,
            "bpr": 0.2,
            "popularity": 0.3
        }

    assert np.isclose(
        sum(weights.values()),
        1.0
    )

    candidates = (
        generate_hybrid_candidates(
            user_id
        )
    )

    candidate_ids = (
        candidates["candidate_ids"]
    )

    if not candidate_ids:

        return pd.DataFrame()

    scored = compute_component_scores(
        user_id,
        candidate_ids
    )

    scored["hybrid_score"] = (
        weights["semantic"]
        * scored["semantic_score"]

        + weights["bpr"]
        * scored["bpr_score"]

        + weights["popularity"]
        * scored["popularity_score"]
    )

    scored = (
        scored.sort_values(
            "hybrid_score",
            ascending=False,
            kind="stable"
        )
        .head(k)
        .copy()
    )

    # Add product information
    display_cols = [
        "product_id",
        "product_name",
        "source",
        "price",
        "currency",
        "average_rating"
    ]

    available_cols = [
        col
        for col in display_cols
        if col in product_catalog.columns
    ]

    scored = scored.merge(
        product_catalog[
            available_cols
        ],
        on="product_id",
        how="left",
        validate="one_to_one"
    )

    return scored.reset_index(
        drop=True
    )


# ------------------------------------------------------------
# Sanity test: 5 users
# ------------------------------------------------------------

sample_users = (
    validation["user_id"]
    .head(5)
    .tolist()
)

for user_id in sample_users:

    print("\n" + "=" * 70)
    print("USER:", user_id)
    print("=" * 70)

    result = recommend_hybrid(
        user_id,
        k=5
    )

    display(
        result[
            [
                col
                for col in [
                    "product_id",
                    "product_name",
                    "semantic_score",
                    "bpr_score",
                    "popularity_score",
                    "hybrid_score"
                ]
                if col in result.columns
            ]
        ].round(5)
    )

    seen = user_history.get(
        user_id,
        set()
    )

    assert not set(
        result["product_id"]
    ).intersection(seen)

    assert result[
        "hybrid_score"
    ].notna().all()

print(
    "\n✅ Hybrid sanity test passed"
)


USER: amazon_AE22DOLTN2WN6B3TJ32UDXOEFNSA


,product_id,product_name,semantic_score,bpr_score,popularity_score,hybrid_score
0,amazon_B01K8B8YA8,Echo Dot (2nd Generation) - Smart speaker with Alexa - White,0.77017,0.67175,1.00000,0.81944
1,amazon_B00X5C22SS,Sony BDPS3500 Blu-ray Player with Wi-Fi (2015 Model),0.78847,0.79300,0.85017,0.80789
2,amazon_B0C2HWSXNL,"Amazon Basics 16-Gauge Speaker Wire Cable, 100 Feet, 4-Pack",0.86643,0.81408,0.70128,0.80641
3,amazon_B0BGNG1294,"Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, Black",0.83167,0.60434,0.88933,0.80350
4,amazon_B00ZVJAF9G,Fire TV Stick with Voice Remote,0.83559,0.84400,0.70965,0.79949



USER: amazon_AE22MRZKBD5MWJLIMGYLLJ67ZWVQ


,product_id,product_name,semantic_score,bpr_score,popularity_score,hybrid_score
0,amazon_B011BRUOMO,"SanDisk Ultra 32GB microSDHC UHS-I Card with Adapter, Silver, Standard Packaging (SDSQUNC-032G-G...",0.78873,0.79469,0.92470,0.83071
1,amazon_B07P9V8GSH,SanDisk Ultra 32GB UHS-I/Class 10 Micro SDHC Memory Card With Adapter - SDSDQUAN-032G-G4A,0.75561,0.82820,0.84538,0.79706
2,amazon_B00X5C22SS,Sony BDPS3500 Blu-ray Player with Wi-Fi (2015 Model),0.79337,0.69073,0.85017,0.78988
3,amazon_B000WYVBR0,VideoSecu ML531BE2 TV Wall Mount kit with Free Magnetic Stud Finder and HDMI Cable for Most 26-5...,0.80415,0.71619,0.81022,0.78838
4,amazon_B01K8B8YA8,Echo Dot (2nd Generation) - Smart speaker with Alexa - White,0.68971,0.71249,1.00000,0.78735



USER: amazon_AE22PJ54OVIRX3I6KSLMPRHPHA4A


...


USER: amazon_AE22U2DM2E4EOEDEEZ46BDVIYPOQ


,product_id,product_name,semantic_score,bpr_score,popularity_score,hybrid_score
0,amazon_B011BRUOMO,"SanDisk Ultra 32GB microSDHC UHS-I Card with Adapter, Silver, Standard Packaging (SDSQUNC-032G-G...",0.77906,0.54587,0.92470,0.77611
1,amazon_B07454F4JH,"Seagate Backup Plus Portable 5TB External Hard Drive HDD – Red USB 3.0 for PC Laptop and Mac, 2 ...",0.75186,0.75338,0.79488,0.76507
2,amazon_B07P9V8GSH,SanDisk Ultra 32GB UHS-I/Class 10 Micro SDHC Memory Card With Adapter - SDSDQUAN-032G-G4A,0.74907,0.63449,0.84538,0.75505
3,amazon_B0BGNG1294,"Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, Black",0.71735,0.63929,0.88933,0.75333
4,amazon_B01K8B8YA8,Echo Dot (2nd Generation) - Smart speaker with Alexa - White,0.69433,0.52398,1.00000,0.75196



USER: amazon_AE22Z3RLVIRU6RT5PNRK5CFFNEFQ


,product_id,product_name,semantic_score,bpr_score,popularity_score,hybrid_score
0,amazon_B07S764D9V,"Panasonic ErgoFit Wired Earbuds, In-Ear Headphones with Microphone and Call Controller, Ergonomi...",0.84134,0.39495,0.94580,0.78340
1,amazon_B01K8B8YA8,Echo Dot (2nd Generation) - Smart speaker with Alexa - White,0.76208,0.30110,1.00000,0.74126
2,amazon_B0BLHNNSGG,"Logitech MX Keys Mini Minimalist Wireless Illuminated Keyboard, Compact, Bluetooth, Backlit, USB...",0.80015,0.90656,0.52600,0.73919
3,amazon_B00DS4G2AW,"Amazon Basics 11.6-Inch Laptop Sleeve, Protective Case with Zipper - Black",0.82615,0.51105,0.74274,0.73811
4,amazon_B07VHBC8VP,PALOVUE Lightning Headphones Earphones Earbuds in-Ear Magnetic MFi Certified with Microphone Com...,0.82017,0.97990,0.43326,0.73604



✅ Hybrid sanity test passed


# Candidate Coverage Audit

In [22]:
# ============================================================
# STEP 7 - CELL 17
# CANDIDATE COVERAGE AUDIT
# ============================================================

print("=" * 70)
print("HYBRID CANDIDATE COVERAGE AUDIT")
print("=" * 70)

SAMPLE_SIZE = 500

validation_sample = validation.sample(
    n=min(SAMPLE_SIZE, len(validation)),
    random_state=RANDOM_STATE
).copy()

coverage_rows = []

start = time.time()

for row in validation_sample.itertuples():

    candidates = generate_hybrid_candidates(
        row.user_id
    )

    target = row.product_id

    coverage_rows.append({
        "user_id": row.user_id,
        "target": target,

        "semantic_hit":
            target in candidates["semantic_ids"],

        "bpr_hit":
            target in candidates["bpr_ids"],

        "popularity_hit":
            target in candidates["popularity_ids"],

        "union_hit":
            target in candidates["candidate_ids"],

        "candidate_count":
            len(candidates["candidate_ids"])
    })

candidate_coverage = pd.DataFrame(
    coverage_rows
)

print("\nUsers evaluated:",
      f"{len(candidate_coverage):,}")

print("\nCANDIDATE COVERAGE")

for col in [
    "semantic_hit",
    "bpr_hit",
    "popularity_hit",
    "union_hit"
]:

    print(
        f"{col:<20}: "
        f"{candidate_coverage[col].mean()*100:.2f}%"
    )

print("\nCandidate count:")

display(
    candidate_coverage[
        "candidate_count"
    ].describe()
)

print(
    "\nExecution time:",
    f"{time.time()-start:.2f} sec"
)

HYBRID CANDIDATE COVERAGE AUDIT

Users evaluated: 500

CANDIDATE COVERAGE
semantic_hit        : 0.60%
bpr_hit             : 4.20%
popularity_hit      : 16.60%
union_hit           : 20.00%

Candidate count:


count    500.000000
mean     591.026000
std        6.023112
min      569.000000
25%      588.000000
50%      592.000000
75%      595.250000
max      600.000000
Name: candidate_count, dtype: float64


Execution time: 17.93 sec


In [23]:
# ============================================================
# STEP 7 - CELL 18
# HYBRID VALIDATION - 500 USERS
# ============================================================

HYBRID_CONFIGS = {
    "SemanticOnly": (1.0, 0.0, 0.0),
    "BPROnly": (0.0, 1.0, 0.0),
    "PopularityOnly": (0.0, 0.0, 1.0),

    "Hybrid_A": (0.5, 0.2, 0.3),
    "Hybrid_B": (0.6, 0.1, 0.3),
    "Hybrid_C": (0.7, 0.1, 0.2),
    "Hybrid_D": (0.4, 0.1, 0.5)
}

KS = [5, 10, 20]

metric_store = {
    name: {
        k: {
            "hit": [],
            "ndcg": [],
            "mrr": []
        }
        for k in KS
    }
    for name in HYBRID_CONFIGS
}

start = time.time()

for row in validation_sample.itertuples():

    candidates = generate_hybrid_candidates(
        row.user_id
    )

    candidate_ids = candidates[
        "candidate_ids"
    ]

    if not candidate_ids:
        continue

    scores = compute_component_scores(
        row.user_id,
        candidate_ids
    )

    target = row.product_id

    for name, weights in HYBRID_CONFIGS.items():

        w_sem, w_bpr, w_pop = weights

        hybrid_scores = (
            w_sem * scores["semantic_score"].to_numpy()
            +
            w_bpr * scores["bpr_score"].to_numpy()
            +
            w_pop * scores["popularity_score"].to_numpy()
        )

        order = np.argsort(
            -hybrid_scores,
            kind="stable"
        )

        ranked_ids = (
            scores["product_id"]
            .to_numpy()[order]
        )

        for k in KS:

            top_k = ranked_ids[:k]

            if target in top_k:

                rank = (
                    np.where(
                        top_k == target
                    )[0][0] + 1
                )

                hit = 1.0

                ndcg = (
                    1.0
                    /
                    np.log2(rank + 1)
                )

                mrr = (
                    1.0 / rank
                )

            else:

                hit = 0.0
                ndcg = 0.0
                mrr = 0.0

            metric_store[name][k]["hit"].append(
                hit
            )

            metric_store[name][k]["ndcg"].append(
                ndcg
            )

            metric_store[name][k]["mrr"].append(
                mrr
            )


rows = []

for name in HYBRID_CONFIGS:

    for k in KS:

        values = metric_store[name][k]

        rows.append({
            "model": name,
            "K": k,
            "users": len(values["hit"]),

            "HitRate":
                np.mean(values["hit"]),

            "Recall":
                np.mean(values["hit"]),

            "NDCG":
                np.mean(values["ndcg"]),

            "MRR":
                np.mean(values["mrr"])
        })

hybrid_sample_results = pd.DataFrame(
    rows
)

print("=" * 70)
print("HYBRID VALIDATION - SAMPLE")
print("=" * 70)

display(
    hybrid_sample_results[
        hybrid_sample_results["K"] == 10
    ]
    .sort_values(
        "NDCG",
        ascending=False
    )
    .round(6)
)

print(
    "\nExecution time:",
    f"{time.time()-start:.2f} sec"
)

HYBRID VALIDATION - SAMPLE


,model,K,users,HitRate,Recall,NDCG,MRR
7,PopularityOnly,10,500,0.030,0.030,0.013646,0.008758
10,Hybrid_A,10,500,0.024,0.024,0.012984,0.009586
19,Hybrid_D,10,500,0.022,0.022,0.011600,0.008286
13,Hybrid_B,10,500,0.022,0.022,0.011124,0.007689
16,Hybrid_C,10,500,0.022,0.022,0.009861,0.006258
4,BPROnly,10,500,0.004,0.004,0.001439,0.000700
1,SemanticOnly,10,500,0.000,0.000,0.000000,0.000000



Execution time: 21.30 sec


In [24]:
# ============================================================
# STEP 7 - CELL 19
# POPULARITY-ANCHORED HYBRID VALIDATION
# ============================================================

print("=" * 70)
print("POPULARITY-ANCHORED HYBRID")
print("=" * 70)

ANCHOR_CONFIGS = {
    "Popularity": (0.0, 0.0, 1.0),

    "PopSem_05": (0.05, 0.0, 0.95),
    "PopSem_10": (0.10, 0.0, 0.90),
    "PopSem_20": (0.20, 0.0, 0.80),

    "PopBPR_05": (0.0, 0.05, 0.95),
    "PopBPR_10": (0.0, 0.10, 0.90),

    "PopSemBPR_A": (0.05, 0.05, 0.90),
    "PopSemBPR_B": (0.10, 0.05, 0.85),
    "PopSemBPR_C": (0.10, 0.10, 0.80)
}

anchor_store = {
    name: {
        k: {
            "hit": [],
            "ndcg": [],
            "mrr": []
        }
        for k in KS
    }
    for name in ANCHOR_CONFIGS
}

start = time.time()

for row in validation_sample.itertuples():

    user_id = row.user_id
    target = row.product_id

    # --------------------------------------------------------
    # Fixed popularity candidate pool
    # --------------------------------------------------------

    candidate_ids = get_popularity_candidates(
        user_id,
        n=200
    )

    scores = compute_component_scores(
        user_id,
        candidate_ids
    )

    sem = scores[
        "semantic_score"
    ].to_numpy()

    bpr = scores[
        "bpr_score"
    ].to_numpy()

    pop = scores[
        "popularity_score"
    ].to_numpy()

    ids = scores[
        "product_id"
    ].to_numpy()

    # --------------------------------------------------------
    # Evaluate configurations
    # --------------------------------------------------------

    for name, weights in ANCHOR_CONFIGS.items():

        w_sem, w_bpr, w_pop = weights

        final_scores = (
            w_sem * sem
            +
            w_bpr * bpr
            +
            w_pop * pop
        )

        order = np.argsort(
            -final_scores,
            kind="stable"
        )

        ranked_ids = ids[order]

        for k in KS:

            top_k = ranked_ids[:k]

            if target in top_k:

                rank = (
                    np.where(
                        top_k == target
                    )[0][0] + 1
                )

                hit = 1.0

                ndcg = (
                    1.0
                    /
                    np.log2(rank + 1)
                )

                mrr = (
                    1.0 / rank
                )

            else:

                hit = 0.0
                ndcg = 0.0
                mrr = 0.0

            anchor_store[name][k]["hit"].append(
                hit
            )

            anchor_store[name][k]["ndcg"].append(
                ndcg
            )

            anchor_store[name][k]["mrr"].append(
                mrr
            )


anchor_rows = []

for name in ANCHOR_CONFIGS:

    for k in KS:

        values = anchor_store[name][k]

        anchor_rows.append({
            "model": name,
            "K": k,
            "users": len(values["hit"]),

            "HitRate":
                np.mean(values["hit"]),

            "Recall":
                np.mean(values["hit"]),

            "NDCG":
                np.mean(values["ndcg"]),

            "MRR":
                np.mean(values["mrr"])
        })


anchor_results = pd.DataFrame(
    anchor_rows
)

print("\n" + "=" * 70)
print("POPULARITY-ANCHORED VALIDATION @10")
print("=" * 70)

display(
    anchor_results[
        anchor_results["K"] == 10
    ]
    .sort_values(
        "NDCG",
        ascending=False
    )
    .round(6)
)

print(
    "\nExecution time:",
    f"{time.time()-start:.2f} sec"
)

POPULARITY-ANCHORED HYBRID

POPULARITY-ANCHORED VALIDATION @10


,model,K,users,HitRate,Recall,NDCG,MRR
16,PopBPR_10,10,500,0.030,0.030,0.014072,0.009208
4,PopSem_05,10,500,0.030,0.030,0.013728,0.008841
1,Popularity,10,500,0.030,0.030,0.013646,0.008758
7,PopSem_10,10,500,0.028,0.028,0.013233,0.008733
13,PopBPR_05,10,500,0.028,0.028,0.013228,0.008728
19,PopSemBPR_A,10,500,0.028,0.028,0.013204,0.008706
25,PopSemBPR_C,10,500,0.026,0.026,0.012746,0.008644
22,PopSemBPR_B,10,500,0.026,0.026,0.012602,0.008483
10,PopSem_20,10,500,0.026,0.026,0.012541,0.008417



Execution time: 0.87 sec


In [25]:
# ============================================================
# STEP 7 - CELL 20
# TOP-10 RANKING CHANGE AUDIT
# ============================================================

print("=" * 70)
print("TOP-10 RANKING CHANGE AUDIT")
print("=" * 70)

sample_users = (
    validation_sample["user_id"]
    .head(100)
    .tolist()
)

change_rows = []

for user_id in sample_users:

    candidate_ids = get_popularity_candidates(
        user_id,
        n=200
    )

    scores = compute_component_scores(
        user_id,
        candidate_ids
    )

    sem = scores["semantic_score"].to_numpy()
    bpr = scores["bpr_score"].to_numpy()
    pop = scores["popularity_score"].to_numpy()

    ids = scores["product_id"].to_numpy()

    baseline_order = np.argsort(
        -pop,
        kind="stable"
    )

    baseline_top10 = set(
        ids[baseline_order[:10]]
    )

    for name, weights in ANCHOR_CONFIGS.items():

        w_sem, w_bpr, w_pop = weights

        final_scores = (
            w_sem * sem
            +
            w_bpr * bpr
            +
            w_pop * pop
        )

        order = np.argsort(
            -final_scores,
            kind="stable"
        )

        hybrid_top10 = set(
            ids[order[:10]]
        )

        overlap = len(
            baseline_top10.intersection(
                hybrid_top10
            )
        )

        change_rows.append({
            "model": name,
            "top10_overlap": overlap,
            "top10_changed": 10 - overlap
        })


change_results = (
    pd.DataFrame(change_rows)
    .groupby("model")
    .agg(
        avg_top10_overlap=(
            "top10_overlap",
            "mean"
        ),

        avg_top10_changed=(
            "top10_changed",
            "mean"
        )
    )
    .reset_index()
)

display(
    change_results.sort_values(
        "avg_top10_changed",
        ascending=False
    ).round(3)
)

TOP-10 RANKING CHANGE AUDIT


,model,avg_top10_overlap,avg_top10_changed
4,PopSemBPR_C,8.60,1.40
7,PopSem_20,8.90,1.10
1,PopBPR_10,8.97,1.03
3,PopSemBPR_B,9.26,0.74
2,PopSemBPR_A,9.50,0.50
0,PopBPR_05,9.67,0.33
6,PopSem_10,9.79,0.21
5,PopSem_05,9.96,0.04
8,Popularity,10.00,0.00


In [26]:
# ============================================================
# STEP 7 - CELL 21
# FULL POPULARITY-ANCHORED VALIDATION
# ============================================================

import time
import numpy as np
import pandas as pd

print("=" * 70)
print("FULL POPULARITY-ANCHORED VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

FULL_KS = [5, 10, 20]

FULL_CONFIGS = {
    "Popularity": (0.0, 0.0, 1.0),

    "PopSem_05": (0.05, 0.0, 0.95),
    "PopSem_10": (0.10, 0.0, 0.90),
    "PopSem_20": (0.20, 0.0, 0.80),

    "PopBPR_05": (0.0, 0.05, 0.95),
    "PopBPR_10": (0.0, 0.10, 0.90),

    "PopSemBPR_A": (0.05, 0.05, 0.90),
    "PopSemBPR_B": (0.10, 0.05, 0.85),
    "PopSemBPR_C": (0.10, 0.10, 0.80)
}

# ------------------------------------------------------------
# 2. Evaluation population
# ------------------------------------------------------------

full_validation = (
    validation
    .reset_index(drop=True)
    .copy()
)

assert full_validation["user_id"].is_unique

print(
    "Validation users:",
    f"{len(full_validation):,}"
)

print(
    "Configurations:",
    len(FULL_CONFIGS)
)

print(
    "Candidate pool:",
    POPULARITY_CANDIDATES
)

# ------------------------------------------------------------
# 3. Metric storage
# ------------------------------------------------------------

metric_store_full = {
    name: {
        k: {
            "hit": 0,
            "ndcg": 0.0,
            "mrr": 0.0
        }
        for k in FULL_KS
    }
    for name in FULL_CONFIGS
}

# Per-user results for paired comparison
per_user_results = []

candidate_hits = 0

start = time.time()

# ------------------------------------------------------------
# 4. Evaluate every validation user
# ------------------------------------------------------------

for i, row in enumerate(
    full_validation.itertuples(index=False),
    start=1
):

    user_id = row.user_id
    target = row.product_id

    candidate_ids = get_popularity_candidates(
        user_id,
        n=POPULARITY_CANDIDATES
    )

    candidate_hits += int(
        target in candidate_ids
    )

    scores = compute_component_scores(
        user_id,
        candidate_ids
    )

    sem = scores[
        "semantic_score"
    ].to_numpy(dtype=np.float32)

    bpr = scores[
        "bpr_score"
    ].to_numpy(dtype=np.float32)

    pop = scores[
        "popularity_score"
    ].to_numpy(dtype=np.float32)

    ids = scores[
        "product_id"
    ].to_numpy()

    # --------------------------------------------------------
    # Evaluate all weight configurations
    # --------------------------------------------------------

    for name, weights in FULL_CONFIGS.items():

        w_sem, w_bpr, w_pop = weights

        final_scores = (
            w_sem * sem
            + w_bpr * bpr
            + w_pop * pop
        )

        order = np.argsort(
            -final_scores,
            kind="stable"
        )

        ranked_ids = ids[order]

        target_positions = np.flatnonzero(
            ranked_ids[:max(FULL_KS)] == target
        )

        if len(target_positions) > 0:

            rank = int(
                target_positions[0] + 1
            )

        else:

            rank = None

        user_result = {
            "user_id": user_id,
            "product_id": target,
            "model": name,
            "rank": rank,
            "candidate_hit": (
                target in candidate_ids
            )
        }

        for k in FULL_KS:

            hit = (
                rank is not None
                and rank <= k
            )

            ndcg = (
                1.0 / np.log2(rank + 1)
                if hit
                else 0.0
            )

            mrr = (
                1.0 / rank
                if hit
                else 0.0
            )

            metric_store_full[
                name
            ][k]["hit"] += int(hit)

            metric_store_full[
                name
            ][k]["ndcg"] += ndcg

            metric_store_full[
                name
            ][k]["mrr"] += mrr

            user_result[
                f"hit@{k}"
            ] = int(hit)

            user_result[
                f"ndcg@{k}"
            ] = ndcg

        per_user_results.append(
            user_result
        )

    if i % 5000 == 0:

        print(
            f"Processed {i:,}/"
            f"{len(full_validation):,} users"
        )

# ------------------------------------------------------------
# 5. Aggregate metrics
# ------------------------------------------------------------

n_users = len(
    full_validation
)

full_rows = []

for name in FULL_CONFIGS:

    for k in FULL_KS:

        values = (
            metric_store_full[
                name
            ][k]
        )

        full_rows.append({
            "model": name,
            "K": k,
            "users": n_users,

            "hits":
                values["hit"],

            "HitRate":
                values["hit"] / n_users,

            "Recall":
                values["hit"] / n_users,

            "NDCG":
                values["ndcg"] / n_users,

            "MRR":
                values["mrr"] / n_users
        })

full_anchor_results = pd.DataFrame(
    full_rows
)

full_anchor_user_results = pd.DataFrame(
    per_user_results
)

# ------------------------------------------------------------
# 6. Output
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FULL VALIDATION - CANDIDATE COVERAGE")
print("=" * 70)

print(
    "Candidate hits:",
    f"{candidate_hits:,}"
)

print(
    "Candidate coverage:",
    f"{candidate_hits/n_users*100:.2f}%"
)

print("\n" + "=" * 70)
print("FULL VALIDATION @10")
print("=" * 70)

display(
    full_anchor_results[
        full_anchor_results["K"] == 10
    ]
    .sort_values(
        "NDCG",
        ascending=False
    )
    .round(6)
)

print(
    "\nExecution time:",
    f"{time.time()-start:.2f} sec"
)

FULL POPULARITY-ANCHORED VALIDATION
Validation users: 26,903
Configurations: 9
Candidate pool: 200
Processed 5,000/26,903 users
Processed 10,000/26,903 users
Processed 15,000/26,903 users
Processed 20,000/26,903 users
Processed 25,000/26,903 users

FULL VALIDATION - CANDIDATE COVERAGE
Candidate hits: 3,929
Candidate coverage: 14.60%

FULL VALIDATION @10


,model,K,users,hits,HitRate,Recall,NDCG,MRR
25,PopSemBPR_C,10,26903,717,0.026651,0.026651,0.013594,0.009692
16,PopBPR_10,10,26903,721,0.026800,0.026800,0.013499,0.009522
22,PopSemBPR_B,10,26903,717,0.026651,0.026651,0.013402,0.009452
19,PopSemBPR_A,10,26903,716,0.026614,0.026614,0.013376,0.009426
13,PopBPR_05,10,26903,714,0.026540,0.026540,0.013339,0.009400
7,PopSem_10,10,26903,705,0.026205,0.026205,0.013183,0.009305
4,PopSem_05,10,26903,702,0.026094,0.026094,0.013138,0.009282
1,Popularity,10,26903,700,0.026019,0.026019,0.013102,0.009260
10,PopSem_20,10,26903,685,0.025462,0.025462,0.012960,0.009226



Execution time: 44.99 sec


In [27]:
# ============================================================
# STEP 7 - CELL 22
# PAIRED COMPARISON AGAINST POPULARITY
# ============================================================

print("=" * 70)
print("PAIRED VALIDATION COMPARISON @10")
print("=" * 70)

baseline = (
    full_anchor_user_results[
        full_anchor_user_results["model"]
        == "Popularity"
    ][
        [
            "user_id",
            "hit@10",
            "ndcg@10"
        ]
    ]
    .rename(
        columns={
            "hit@10": "baseline_hit",
            "ndcg@10": "baseline_ndcg"
        }
    )
)

comparison_rows = []

for name in FULL_CONFIGS:

    if name == "Popularity":
        continue

    model_data = (
        full_anchor_user_results[
            full_anchor_user_results["model"]
            == name
        ][
            [
                "user_id",
                "hit@10",
                "ndcg@10"
            ]
        ]
    )

    paired = model_data.merge(
        baseline,
        on="user_id",
        how="inner",
        validate="one_to_one"
    )

    assert len(paired) == len(
        full_validation
    )

    ndcg_diff = (
        paired["ndcg@10"]
        - paired["baseline_ndcg"]
    )

    comparison_rows.append({
        "model": name,

        "improved_users":
            int((ndcg_diff > 0).sum()),

        "worse_users":
            int((ndcg_diff < 0).sum()),

        "unchanged_users":
            int((ndcg_diff == 0).sum()),

        "extra_hits":
            int(
                (
                    (paired["hit@10"] == 1)
                    &
                    (paired["baseline_hit"] == 0)
                ).sum()
            ),

        "lost_hits":
            int(
                (
                    (paired["hit@10"] == 0)
                    &
                    (paired["baseline_hit"] == 1)
                ).sum()
            ),

        "delta_ndcg":
            float(ndcg_diff.mean())
    })

paired_comparison = pd.DataFrame(
    comparison_rows
)

display(
    paired_comparison.sort_values(
        "delta_ndcg",
        ascending=False
    ).round(7)
)

PAIRED VALIDATION COMPARISON @10


,model,improved_users,worse_users,unchanged_users,extra_hits,lost_hits,delta_ndcg
7,PopSemBPR_C,261,146,26496,89,72,0.000492
4,PopBPR_10,232,122,26549,68,47,0.000397
6,PopSemBPR_B,183,111,26609,54,37,0.000300
5,PopSemBPR_A,164,83,26656,37,21,0.000274
3,PopBPR_05,139,63,26701,26,12,0.000237
1,PopSem_10,70,47,26786,14,9,0.000081
0,PopSem_05,24,7,26872,2,0,0.000036
2,PopSem_20,151,162,26590,53,68,-0.000142


# Khóa cấu hình Hybrid

In [28]:
# ============================================================
# STEP 7 - CELL 23
# LOCK HYBRID CONFIGURATIONS
# ============================================================

import json
from pathlib import Path

STEP7_OUTPUT = Path(
    "/kaggle/working/step7"
)

STEP7_OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_CONFIGS = {

    "Popularity": {
        "semantic": 0.0,
        "bpr": 0.0,
        "popularity": 1.0
    },

    "PopBPR_10": {
        "semantic": 0.0,
        "bpr": 0.1,
        "popularity": 0.9
    },

    "PopSemBPR_C": {
        "semantic": 0.1,
        "bpr": 0.1,
        "popularity": 0.8
    }
}

FINAL_KS = [5, 10, 20]

FINAL_CANDIDATE_COUNT = 200

config = {
    "selection_dataset": "validation",
    "evaluation_dataset": "test",

    "candidate_strategy":
        "train_only_popularity_top_200",

    "candidate_count":
        FINAL_CANDIDATE_COUNT,

    "models":
        FINAL_CONFIGS,

    "evaluation_ks":
        FINAL_KS,

    "selection_metrics": [
        "Recall@10",
        "NDCG@10"
    ]
}

config_path = (
    STEP7_OUTPUT
    / "hybrid_config.json"
)

with open(
    config_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        ensure_ascii=False,
        indent=4
    )

print("=" * 70)
print("FINAL HYBRID CONFIGURATION")
print("=" * 70)

print(
    json.dumps(
        config,
        ensure_ascii=False,
        indent=4
    )
)

print(
    "\nSaved:",
    config_path
)

FINAL HYBRID CONFIGURATION
{
    "selection_dataset": "validation",
    "evaluation_dataset": "test",
    "candidate_strategy": "train_only_popularity_top_200",
    "candidate_count": 200,
    "models": {
        "Popularity": {
            "semantic": 0.0,
            "bpr": 0.0,
            "popularity": 1.0
        },
        "PopBPR_10": {
            "semantic": 0.0,
            "bpr": 0.1,
            "popularity": 0.9
        },
        "PopSemBPR_C": {
            "semantic": 0.1,
            "bpr": 0.1,
            "popularity": 0.8
        }
    },
    "evaluation_ks": [
        5,
        10,
        20
    ],
    "selection_metrics": [
        "Recall@10",
        "NDCG@10"
    ]
}

Saved: /kaggle/working/step7/hybrid_config.json


In [29]:
# ============================================================
# STEP 7 - CELL 24
# FINAL TEST EVALUATION
# ============================================================

import time
import numpy as np
import pandas as pd

print("=" * 70)
print("FINAL HYBRID TEST EVALUATION")
print("=" * 70)

test_eval = (
    test
    .reset_index(drop=True)
    .copy()
)

assert test_eval["user_id"].is_unique

assert len(test_eval) == 26903

test_metric_store = {

    model_name: {

        k: {
            "hit": 0,
            "ndcg": 0.0,
            "mrr": 0.0
        }

        for k in FINAL_KS
    }

    for model_name in FINAL_CONFIGS
}

test_user_rows = []

candidate_hits = 0

start = time.time()

for i, row in enumerate(
    test_eval.itertuples(index=False),
    start=1
):

    user_id = row.user_id
    target = row.product_id

    # --------------------------------------------------------
    # 1. Train-only popularity candidates
    # --------------------------------------------------------

    candidate_ids = (
        get_popularity_candidates(
            user_id,
            n=FINAL_CANDIDATE_COUNT
        )
    )

    target_in_candidates = (
        target in candidate_ids
    )

    candidate_hits += int(
        target_in_candidates
    )

    # --------------------------------------------------------
    # 2. Component scores
    # --------------------------------------------------------

    scores = compute_component_scores(
        user_id,
        candidate_ids
    )

    sem = scores[
        "semantic_score"
    ].to_numpy(dtype=np.float32)

    bpr = scores[
        "bpr_score"
    ].to_numpy(dtype=np.float32)

    pop = scores[
        "popularity_score"
    ].to_numpy(dtype=np.float32)

    ids = scores[
        "product_id"
    ].to_numpy()

    # --------------------------------------------------------
    # 3. Evaluate locked models
    # --------------------------------------------------------

    for model_name, weights in (
        FINAL_CONFIGS.items()
    ):

        final_scores = (

            weights["semantic"] * sem

            + weights["bpr"] * bpr

            + weights["popularity"] * pop

        )

        order = np.argsort(
            -final_scores,
            kind="stable"
        )

        ranked_ids = ids[order]

        target_positions = (
            np.flatnonzero(
                ranked_ids[
                    :max(FINAL_KS)
                ] == target
            )
        )

        if len(target_positions) > 0:

            rank = int(
                target_positions[0] + 1
            )

        else:

            rank = None

        user_result = {

            "user_id":
                user_id,

            "product_id":
                target,

            "model":
                model_name,

            "rank":
                rank,

            "candidate_hit":
                target_in_candidates
        }

        for k in FINAL_KS:

            hit = (
                rank is not None
                and rank <= k
            )

            ndcg = (
                1.0 / np.log2(rank + 1)
                if hit
                else 0.0
            )

            mrr = (
                1.0 / rank
                if hit
                else 0.0
            )

            test_metric_store[
                model_name
            ][k]["hit"] += int(hit)

            test_metric_store[
                model_name
            ][k]["ndcg"] += ndcg

            test_metric_store[
                model_name
            ][k]["mrr"] += mrr

            user_result[
                f"hit@{k}"
            ] = int(hit)

            user_result[
                f"ndcg@{k}"
            ] = ndcg

            user_result[
                f"mrr@{k}"
            ] = mrr

        test_user_rows.append(
            user_result
        )

    if i % 5000 == 0:

        print(
            f"Processed {i:,}/"
            f"{len(test_eval):,}"
        )

# ------------------------------------------------------------
# 4. Aggregate results
# ------------------------------------------------------------

n_test_users = len(
    test_eval
)

test_rows = []

for model_name in FINAL_CONFIGS:

    for k in FINAL_KS:

        values = (
            test_metric_store[
                model_name
            ][k]
        )

        test_rows.append({

            "model":
                model_name,

            "K":
                k,

            "users":
                n_test_users,

            "hits":
                values["hit"],

            "HitRate":
                values["hit"]
                / n_test_users,

            "Recall":
                values["hit"]
                / n_test_users,

            "NDCG":
                values["ndcg"]
                / n_test_users,

            "MRR":
                values["mrr"]
                / n_test_users
        })

final_test_results = pd.DataFrame(
    test_rows
)

final_test_user_results = pd.DataFrame(
    test_user_rows
)

# ------------------------------------------------------------
# 5. Display
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TEST CANDIDATE COVERAGE")
print("=" * 70)

print(
    "Test users:",
    f"{n_test_users:,}"
)

print(
    "Candidate hits:",
    f"{candidate_hits:,}"
)

print(
    "Candidate coverage:",
    f"{candidate_hits/n_test_users*100:.2f}%"
)

print("\n" + "=" * 70)
print("FINAL TEST @10")
print("=" * 70)

display(
    final_test_results[
        final_test_results["K"] == 10
    ].round(6)
)

print("\n" + "=" * 70)
print("FINAL TEST @5 / @10 / @20")
print("=" * 70)

display(
    final_test_results.round(6)
)

print(
    "\nExecution time:",
    f"{time.time()-start:.2f} sec"
)

FINAL HYBRID TEST EVALUATION
Processed 5,000/26,903
Processed 10,000/26,903
Processed 15,000/26,903
Processed 20,000/26,903
Processed 25,000/26,903

TEST CANDIDATE COVERAGE
Test users: 26,903
Candidate hits: 3,342
Candidate coverage: 12.42%

FINAL TEST @10


,model,K,users,hits,HitRate,Recall,NDCG,MRR
1,Popularity,10,26903,497,0.018474,0.018474,0.009149,0.006404
4,PopBPR_10,10,26903,500,0.018585,0.018585,0.009250,0.006484
7,PopSemBPR_C,10,26903,485,0.018028,0.018028,0.009081,0.006421



FINAL TEST @5 / @10 / @20


,model,K,users,hits,HitRate,Recall,NDCG,MRR
0,Popularity,5,26903,269,0.009999,0.009999,0.006504,0.005368
1,Popularity,10,26903,497,0.018474,0.018474,0.009149,0.006404
2,Popularity,20,26903,722,0.026837,0.026837,0.011256,0.006979
3,PopBPR_10,5,26903,278,0.010333,0.010333,0.006634,0.005436
4,PopBPR_10,10,26903,500,0.018585,0.018585,0.009250,0.006484
5,PopBPR_10,20,26903,779,0.028956,0.028956,0.011853,0.007189
6,PopSemBPR_C,5,26903,278,0.010333,0.010333,0.006634,0.005435
7,PopSemBPR_C,10,26903,485,0.018028,0.018028,0.009081,0.006421
8,PopSemBPR_C,20,26903,794,0.029513,0.029513,0.011969,0.007205



Execution time: 34.45 sec


In [31]:
# ============================================================
# STEP 7 - CELL 26 (FIXED)
# EMBEDDING TEMPORAL LEAKAGE PRE-AUDIT
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 70)
print("EMBEDDING TEMPORAL LEAKAGE PRE-AUDIT")
print("=" * 70)

INPUT_DIR = Path("/kaggle/input")

# ------------------------------------------------------------
# 1. Find input files
# ------------------------------------------------------------

def find_input_file(filename):

    matches = list(
        INPUT_DIR.rglob(filename)
    )

    if not matches:
        return None

    return matches[0]


reviews_path = find_input_file(
    "reviews.csv"
)

metadata_path = find_input_file(
    "product_embedding_metadata.csv"
)

# ------------------------------------------------------------
# 2. Product columns
# ------------------------------------------------------------

print("\n[1] PRODUCT COLUMNS")

print(
    products.columns.tolist()
)

# ------------------------------------------------------------
# 3. Product embedding metadata
# ------------------------------------------------------------

print("\n[2] PRODUCT EMBEDDING METADATA")

if metadata_path is not None:

    embedding_metadata_audit = pd.read_csv(
        metadata_path,
        nrows=5
    )

    print("File:", metadata_path)

    print(
        "Columns:",
        embedding_metadata_audit.columns.tolist()
    )

else:

    print(
        "product_embedding_metadata.csv not found"
    )

# ------------------------------------------------------------
# 4. Product text fields
# ------------------------------------------------------------

print("\n[3] PRODUCT TEXT FIELDS")

text_columns = [
    col
    for col in products.columns
    if any(
        keyword in col.lower()
        for keyword in [
            "text",
            "review",
            "description",
            "feature",
            "title"
        ]
    )
]

print(text_columns)

# ------------------------------------------------------------
# 5. Review data
# ------------------------------------------------------------

print("\n[4] REVIEW DATA")

if reviews_path is not None:

    print("File:", reviews_path)

    review_columns = pd.read_csv(
        reviews_path,
        nrows=0
    ).columns.tolist()

    print(
        "Review columns:",
        review_columns
    )

    # Count rows without loading ~290 MB into RAM
    with open(
        reviews_path,
        "rb"
    ) as f:

        review_count = (
            sum(
                chunk.count(b"\n")
                for chunk in iter(
                    lambda: f.read(8 * 1024 * 1024),
                    b""
                )
            ) - 1
        )

    print(
        "Reviews:",
        f"{review_count:,}"
    )

else:

    print(
        "reviews.csv not found in Kaggle Input"
    )

# ------------------------------------------------------------
# 6. Product embeddings
# ------------------------------------------------------------

print("\n[5] PRODUCT EMBEDDINGS")

print(
    "Embedding shape:",
    product_embeddings.shape
)

print(
    "Product rows:",
    len(products)
)

assert (
    len(products)
    == len(product_embeddings)
)

print("\n✅ Pre-audit completed")

EMBEDDING TEMPORAL LEAKAGE PRE-AUDIT

[1] PRODUCT COLUMNS
['product_id', 'product_name', 'category', 'subcategory', 'brand', 'price', 'currency', 'average_rating', 'review_count', 'sold_count', 'description', 'image_url', 'product_url', 'source', 'language']

[2] PRODUCT EMBEDDING METADATA
File: /kaggle/input/datasets/trnquangtriu/7dataset/product_embedding_metadata.csv
Columns: ['product_id', 'product_name', 'category', 'subcategory', 'brand', 'price', 'currency', 'average_rating', 'review_count', 'sold_count', 'description', 'image_url', 'product_url', 'source', 'language', 'clean_product_name', 'clean_category', 'clean_subcategory', 'clean_brand', 'clean_description', 'product_text', 'product_text_word_count', 'semantic_usable', 'product_embedding_id']

[3] PRODUCT TEXT FIELDS
['review_count', 'description']

[4] REVIEW DATA
File: /kaggle/input/datasets/trnquangtriu/7dataset/reviews.csv
Review columns: ['review_id', 'user_id', 'product_id', 'rating', 'review_title', 'review_text', '

In [32]:
# ============================================================
# STEP 7 - CELL 27
# PRODUCT EMBEDDING CONTENT AUDIT
# ============================================================

import pandas as pd
from pathlib import Path

print("=" * 70)
print("PRODUCT EMBEDDING CONTENT AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Locate metadata
# ------------------------------------------------------------

metadata_files = list(
    Path("/kaggle/input").rglob(
        "product_embedding_metadata.csv"
    )
)

assert metadata_files, (
    "product_embedding_metadata.csv not found"
)

metadata_path = metadata_files[0]

# ------------------------------------------------------------
# 2. Read sample
# ------------------------------------------------------------

sample_metadata = pd.read_csv(
    metadata_path,
    nrows=1000
)

# ------------------------------------------------------------
# 3. Inspect product text
# ------------------------------------------------------------

columns_to_show = [
    "product_id",
    "source",
    "product_name",
    "category",
    "brand",
    "description",
    "product_text"
]

columns_to_show = [
    c for c in columns_to_show
    if c in sample_metadata.columns
]

for source in ["amazon", "shopee", "tiki"]:

    subset = sample_metadata[
        sample_metadata["source"] == source
    ]

    if subset.empty:
        continue

    print("\n" + "=" * 70)
    print("SOURCE:", source.upper())
    print("=" * 70)

    display(
        subset[
            columns_to_show
        ].head(2).T
    )

# ------------------------------------------------------------
# 4. Inspect semantic text structure
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PRODUCT TEXT SAMPLE")
print("=" * 70)

for text in (
    sample_metadata["product_text"]
    .dropna()
    .head(5)
):

    print("\n" + "-" * 60)

    print(
        str(text)[:1000]
    )

print("\n✅ Product text inspection completed")

PRODUCT EMBEDDING CONTENT AUDIT

SOURCE: AMAZON


,0,1
product_id,amazon_B07PXGQC1Q,amazon_B07KTYJ769
source,amazon,amazon
product_name,Apple AirPods (2nd Generation) Wireless Earbuds with Lightning Charging Case Included. Over 24 H...,"Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Device"
category,Apple Products,Amazon Devices
brand,Apple,Amazon
description,NaN,NaN
product_text,Product: Apple AirPods (2nd Generation) Wireless Earbuds with Lightning Charging Case Included. ...,"Product: Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Devic..."



PRODUCT TEXT SAMPLE

------------------------------------------------------------
Product: Apple AirPods (2nd Generation) Wireless Earbuds with Lightning Charging Case Included. Over 24 Hours of Battery Life, Effortless Setup. Bluetooth Headphones for iPhone | Category: Apple Products | Brand: Apple

------------------------------------------------------------
Product: Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Device | Category: Amazon Devices | Brand: Amazon

------------------------------------------------------------
Product: Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, Black | Category: Home Audio & Theater | Brand: Amazon Basics

------------------------------------------------------------
Product: SanDisk 32GB 2-Pack Ultra MicroSDHC UHS-I Memory Card (2x32GB) - SDSQUAR-032G-GN6MT | Category: Computers | Brand: SanDisk

------------------------------------------------------------
Product: Fire TV Sti

In [33]:
# ============================================================
# STEP 7 - CELL 25
# SAVE FINAL EVALUATION ARTIFACTS
# ============================================================

final_test_results.to_csv(

    STEP7_OUTPUT
    / "hybrid_test_results.csv",

    index=False
)

final_test_user_results.to_csv(

    STEP7_OUTPUT
    / "hybrid_test_user_results.csv",

    index=False
)

full_anchor_results.to_csv(

    STEP7_OUTPUT
    / "hybrid_validation_results.csv",

    index=False
)

paired_comparison.to_csv(

    STEP7_OUTPUT
    / "hybrid_paired_validation.csv",

    index=False
)

print("=" * 70)
print("STEP 7 OUTPUT SAVED")
print("=" * 70)

for path in sorted(
    STEP7_OUTPUT.iterdir()
):

    if path.is_file():

        print(
            f"{path.name:<35}"
            f"{path.stat().st_size / 1024:.2f} KB"
        )

print(
    "\nOutput directory:",
    STEP7_OUTPUT
)

STEP 7 OUTPUT SAVED
hybrid_config.json                 0.66 KB
hybrid_paired_validation.csv       0.49 KB
hybrid_test_results.csv            0.98 KB
hybrid_test_user_results.csv       8117.19 KB
hybrid_validation_results.csv      2.86 KB

Output directory: /kaggle/working/step7
